In [15]:
import requests
import pandas as pd
from datetime import datetime, date, timezone, timedelta
import json
import time
import random
from typing import Any
import json
from pathlib import Path
import re
import xml.etree.ElementTree as ET

from renewables_permitting.utils import as_list, save_parquet, validate_required_columns
import xml.etree.ElementTree as ET

from pydantic_ai import Agent, RunContext
import asyncio
from pydantic import BaseModel, Field
import httpx
import requests
from enum import Enum

BASE_URL = "https://www.boe.es/datosabiertos/api/boe/sumario"

# PROJECT_ROOT = Path(__file__).resolve().parents[2]  # fuera del notebook
PROJECT_ROOT = Path.cwd().parent  # dentro del notebook

DATA_DIR = PROJECT_ROOT / "data"

BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"

BOE_DOCS_XML_DIR = BRONZE_DIR / "boe_docs_xml"

BOE_CANDIDATES_PATH = (
    SILVER_DIR
    / "boe_candidates"
    / "boe_candidates_normalized.parquet"
)

BOE_CANDIDATES_DOCS_TEXT_DIR = (
    SILVER_DIR
    / "boe_candidates_docs_text"
)

BOE_CANDIDATES_DOCS_TEXT_PATH = (
    BOE_CANDIDATES_DOCS_TEXT_DIR
    / "boe_candidates_docs_text.parquet"
)

In [16]:
df = pd.read_parquet(BOE_CANDIDATES_DOCS_TEXT_PATH)

df_test = df.loc[df["xml_status"] == "ok"]

df_test.head(2)

,identificador,doc_file_stem,url_xml,fecha_publicacion,titulo,epigrafe_nombre,departamento_nombre,seccion_nombre,xml_path,texto_limpio,texto_len,xml_status,parse_error,parsed_at
0,BOE-A-2023-10297,20230428_BOE-A-2023-10297,https://www.boe.es/diario_boe/xml.php?id=BOE-A...,2023-04-28,"Resolución de 14 de abril de 2023, de la Direc...",Impacto ambiental,MINISTERIO PARA LA TRANSICIÓN ECOLÓGICA Y EL R...,III. Otras disposiciones,/home/bgonzale/CiDaeN/_15_TrabajoFinMaster/tfm...,BOE-A-2023-10297 Estatal Ministerio para la Tr...,28241,ok,None,2026-06-12T11:25:22.292592+00:00
1,BOE-A-2023-10298,20230428_BOE-A-2023-10298,https://www.boe.es/diario_boe/xml.php?id=BOE-A...,2023-04-28,"Resolución de 17 de abril de 2023, de la Direc...",Instalaciones eléctricas,MINISTERIO PARA LA TRANSICIÓN ECOLÓGICA Y EL R...,III. Otras disposiciones,/home/bgonzale/CiDaeN/_15_TrabajoFinMaster/tfm...,BOE-A-2023-10298 Estatal Ministerio para la Tr...,25477,ok,None,2026-06-12T11:25:22.292592+00:00


In [17]:
for _, row in df_test.sample(10, random_state=42).iterrows():
    print("=" * 80)
    print(row["identificador"])
    print(row["titulo"])
    print(row["texto_limpio"][:6000])

BOE-A-2023-10305
Resolución de 17 de abril de 2023, de la Dirección General de Política Energética y Minas, por la que se otorga a Enel Green Power España, SL, autorización administrativa previa para el parque eólico Moeche de 50,4 MW instalados, y sus infraestructuras de evacuación, ubicados en San Sadurniño, Moeche, As Somozas y As Pontes (A Coruña).
BOE-A-2023-10305 Estatal Ministerio para la Transición Ecológica y el Reto Demográfico Resolución 20230417 Resolución de 17 de abril de 2023, de la Dirección General de Política Energética y Minas, por la que se otorga a Enel Green Power España, SL, autorización administrativa previa para el parque eólico Moeche de 50,4 MW instalados, y sus infraestructuras de evacuación, ubicados en San Sadurniño, Moeche, As Somozas y As Pontes (A Coruña). Boletín Oficial del Estado 20230428 101 3 59053 59059 https://www.boe.es/boe/dias/2023/04/28/pdfs/BOE-A-2023-10305.pdf N N N A Enel Green Power España, SL (en adelante, el promotor) solicitó, con fech

## Contratos de salida

Qué quiero saber de cada publicación?

### Relevancia energética

In [18]:
class RelevanciaEnergetica(str, Enum):
    RELEVANTE = "relevante"
    NO_RELEVANTE = "no_relevante"
    DUDOSO = "dudoso"
    
# relevante:
#   Documento sobre generación eléctrica renovable, almacenamiento, evacuación, subestaciones, líneas eléctricas o autorizaciones ambientales/administrativas asociadas.

# no_relevante:
#   Documento energético genérico, normativo, estadístico, tarifario, presupuestario o no vinculado a un proyecto concreto.

# dudoso:
#   Documento con vocabulario energético, pero sin información suficiente para saber si corresponde a un proyecto tramitado.

### Tecnología

In [19]:
class TechnologyType(str, Enum):
    FOTOVOLTAICA = "fotovoltaica"
    EOLICA = "eolica"
    TERMOSOLAR = "termosolar"
    HIDROELECTRICA = "hidroelectrica"
    GEOTERMICA = "geotermica"
    BIOMASA = "biomasa"
    BIOGAS = "biogas"
    HIDROGENO_VERDE = "hidrogeno_verde"
    OTRA = "otra"
    DESCONOCIDA = "desconocida"

class Technology(BaseModel):
    technology_type: TechnologyType
    installed_power_mw: float | None = None
    peak_power_mwp: float | None = None
    description: str | None = None

In [20]:
class StorageSystem(BaseModel):
    exists: bool = False
    power_mw: float | None = None
    capacity_mwh: float | None = None

In [21]:
class InfrastructureType(str, Enum):
    SUBESTACION = "subestacion"
    LINEA_ELECTRICA = "linea_electrica"
    CENTRO_SECCIONAMIENTO = "centro_seccionamiento"
    INFRAESTRUCTURA_EVACUACION = "infraestructura_evacuacion"
    OTRA = "otra"
    DESCONOCIDA = "desconocida"

class Infrastructure(BaseModel):
    type: InfrastructureType
    name: str | None = None
    voltage_kv: float | None = None
    length_km: float | None = None

In [22]:
class HybridConfiguration(BaseModel):
    is_hybrid: bool = False
    generation_technology_types: list[TechnologyType] = Field(default_factory=list)
    includes_storage: bool = False
    description: str | None = None

### Localización

In [23]:
class MunicipalityLocation(BaseModel):
    municipality: str
    province: str
    autonomous_community: str | None = None   

### Procedimiento y estado

In [24]:
class ProcedureStage(str, Enum):
    INFORMACION_PUBLICA = "informacion_publica"
    DECLARACION_IMPACTO_AMBIENTAL = "declaracion_impacto_ambiental"
    INFORME_DETERMINACION_AFECCION_AMBIENTAL = "informe_determinacion_afeccion_ambiental"
    AUTORIZACION_ADMINISTRATIVA_PREVIA = "autorizacion_administrativa_previa"
    AUTORIZACION_ADMINISTRATIVA_CONSTRUCCION = "autorizacion_administrativa_construccion"
    AUTORIZACION_EXPLOTACION = "autorizacion_explotacion"
    DECLARACION_UTILIDAD_PUBLICA = "declaracion_utilidad_publica"
    EXPROPIACION = "expropiacion"
    LEVANTAMIENTO_ACTAS = "levantamiento_actas"
    MODIFICACION = "modificacion"
    ARCHIVO_EXPEDIENTE = "archivo_expediente"
    DESISTIMIENTO = "desistimiento"
    INADMISION = "inadmision"
    OTRO = "otro"
    NO_CONSTA = "no_consta"

In [25]:
class ProcedureDecision(str, Enum):
    FAVORABLE = "favorable"
    DESFAVORABLE = "desfavorable"

    AUTORIZADO = "autorizado"
    APROBADO = "aprobado"
    DENEGADO = "denegado"

    SOMETIDO_INFORMACION_PUBLICA = "sometido_informacion_publica"
    CONVOCADO = "convocado"

    ARCHIVADO = "archivado"
    DESISTIDO = "desistido"
    INADMITIDO = "inadmitido"

    NO_SOMETIDO_EIA_ORDINARIA = "no_sometido_eia_ordinaria"
    SOMETIDO_EIA_ORDINARIA = "sometido_eia_ordinaria"

    NO_CONSTA = "no_consta"

In [26]:
class ProcedureEvent(BaseModel):
    procedure_stage: ProcedureStage
    decision: ProcedureDecision = ProcedureDecision.NO_CONSTA
    is_main_event: bool = False
    evidence: str | None = None

### Global

In [27]:
class BOEEnergyDocument(BaseModel):
    # Identificación BOE
    identificador_boe: str
    fecha_publicacion: date

    # Clasificación de relevancia
    relevancia_energetica: RelevanciaEnergetica
    es_relevante_para_proyecto: bool
    relevance_reason: str | None = None
    # relevance_confidence: float = Field(ge=0, le=1)

    # Auditoría de extracción
    # extraction_confidence: float = Field(ge=0, le=1)
    extraction_notes: str | None = None

    # Proyecto
    project_name: str | None = None
    project_aliases: list[str] = Field(default_factory=list)
    promoter: str | None = None

    # Localización
    locations: list[MunicipalityLocation] = Field(default_factory=list)

    # Componentes técnicos
    technologies: list[Technology] = Field(default_factory=list)
    storage_systems: list[StorageSystem] = Field(default_factory=list)
    infrastructure: list[Infrastructure] = Field(default_factory=list)

    # Hibridación
    hybridization: HybridConfiguration = Field(default_factory=HybridConfiguration)

    # Procedimiento administrativo
    main_events: list[ProcedureEvent] = Field(default_factory=list)
    procedure_history: list[ProcedureEvent] = Field(default_factory=list)

### Notas para la estimación objetiva de la confianza

In [28]:
# confidence = 1.0

# confidence = 1.0

# if project_name is None:
#     confidence -= 0.2

# if promoter is None:
#     confidence -= 0.1

# if len(main_events) == 0:
#     confidence -= 0.3

# if len(locations) == 0:
#     confidence -= 0.1

# if project_name is None:
#     confidence -= 0.2

# if promoter is None:
#     confidence -= 0.1

# if len(main_events) == 0:
#     confidence -= 0.3

# if len(locations) == 0:
#     confidence -= 0.1

# df[
#     (df["relevance_confidence"] < 0.7)
#     | (df["extraction_confidence"] < 0.7)
# ]

## Crear agente

In [29]:
agent = Agent(
    "google:gemini-2.5-flash",
    output_type=BOEEnergyDocument,
    instructions="""
Eres un extractor de información de documentos del BOE sobre proyectos energéticos.

Extrae solo información explícitamente contenida en el título o en el texto.
No inventes datos.

Distingue entre:
- main_events: hitos principales publicados por este documento BOE. Normalmente aparecen en el título o en la parte resolutiva.
- procedure_history: antecedentes procedimentales mencionados en el texto, pero que no son el objeto principal del documento.

Si un dato no aparece, usa null, lista vacía o no_consta.

Clasifica como relevante solo documentos vinculados a proyectos concretos de generación eléctrica, almacenamiento, evacuación, subestaciones, líneas eléctricas o trámites administrativos/ambientales asociados.
"""
)

## Prueba

In [36]:
df_test2 = df_test.loc[df_test["identificador"] == "BOE-A-2023-10297"]
df_test2.values

array([['BOE-A-2023-10297', '20230428_BOE-A-2023-10297',
        'https://www.boe.es/diario_boe/xml.php?id=BOE-A-2023-10297',
        '2023-04-28',
        'Resolución de 14 de abril de 2023, de la Dirección General de Calidad y Evaluación Ambiental, por la que se formula informe de determinación de afección ambiental del proyecto "Planta fotovoltaica hibridación PE Angostillos", con una potencia instalada de 31,172 MW, a ubicar en Cerrato en Palencia (Castilla y León).',
        'Impacto ambiental',
        'MINISTERIO PARA LA TRANSICIÓN ECOLÓGICA Y EL RETO DEMOGRÁFICO',
        'III. Otras disposiciones',
        '/home/bgonzale/CiDaeN/_15_TrabajoFinMaster/tfm-renewables-permitting-tracker/data/bronze/boe_docs_xml/20230428_BOE-A-2023-10297.xml',
        'BOE-A-2023-10297 Estatal Ministerio para la Transición Ecológica y el Reto Demográfico Resolución 20230414 Resolución de 14 de abril de 2023, de la Dirección General de Calidad y Evaluación Ambiental, por la que se formula informe de

In [ ]:
records = []

for _, row in df_test2.sample(1, random_state=42).iterrows():
    prompt = f"""
Identificador BOE: {row["identificador"]}
Fecha publicación: {row["fecha_publicacion"]}
Título: {row["titulo"]}

Texto:
{row["texto_limpio"][:12000]}
"""

    result = await agent.run(prompt)
    #records.append(result.output.model_dump())

#records
result.output.model_dump_json()